<a href="https://colab.research.google.com/github/sherf1207/AI-Projects/blob/main/Pytorch_%26_Tensorflow_MNIST_(CV).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This project implements a handwritten digit recognition system using the MNIST dataset. It explores and compares deep learning models built with both TensorFlow (Keras) and PyTorch to classify grayscale images of digits (0–9). The images are normalized and flattened before being fed into simple fully connected neural networks. The models are trained to learn patterns in pixel data and predict the correct digit class. The project demonstrates a complete machine learning pipeline including data preprocessing, model building, training, and evaluation, while also highlighting differences between the two major deep learning frameworks.

#Tensorflow implementation

In [12]:
from tensorflow.keras.datasets import mnist
import tensorflow as tf
(x_train, y_train ),(x_test,y_test)= mnist.load_data()
x_train = x_train.reshape(-1,28*28)/255.0
x_test =x_test.reshape(-1,28*28)/255.0

model_tf=tf.keras.Sequential([
tf.keras.layers.Dense(128,activation='relu',input_shape=(28*28,)),
tf.keras.layers.Dense(10,activation='softmax')
])

model_tf.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model_tf.summary()

model_tf.fit(
    x_train, y_train,
    epochs=8,
    batch_size=32,
    validation_split=0.1
)

loss, acc = model_tf.evaluate(x_test, y_test)
print(acc)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9222 - loss: 0.2732 - val_accuracy: 0.9642 - val_loss: 0.1249
Epoch 2/8
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9647 - loss: 0.1201 - val_accuracy: 0.9685 - val_loss: 0.1163
Epoch 3/8
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9754 - loss: 0.0824 - val_accuracy: 0.9722 - val_loss: 0.0871
Epoch 4/8
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9811 - loss: 0.0612 - val_accuracy: 0.9743 - val_loss: 0.0854
Epoch 5/8
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9851 - loss: 0.0473 - val_accuracy: 0.9773 - val_loss: 0.0853
Epoch 6/8
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9885 - loss: 0.0369 - val_accuracy: 0.9780 - val_loss: 0.0779
Epoch 7/8
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9914 - loss: 0.0286 - val_accuracy: 0.9787 - val_loss: 0.0726
Epoch 8/8
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9928 - loss: 0.0239 - val_accu

#Pytorch implementation

In [13]:
import torch.nn as nn
import torch.optim as optim
import torch
from torch.utils.data import DataLoader, TensorDataset

x_train_tensor=torch.Tensor(x_train)
y_train_tensor=torch.Tensor(y_train).long()
x_test_tensor=torch.Tensor(x_test)
y_test_tensor=torch.Tensor(y_test).long()

#Combining inputs and labels in one dataset
train_data=TensorDataset(x_train_tensor,y_train_tensor)
train_loader=DataLoader(train_data,batch_size=32,shuffle=True)

class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN ,self).__init__()
        self.fc1=nn.Linear(28*28,128)
        self.fc2=nn.Linear(128,10)

    def forward(self, x):
        x=torch.relu(self.fc1(x))
        x=self.fc2(x) #it uses softmax automatically
        return x

model_torch=SimpleNN()
print("\nPyTorch Model Architecture:")
print(model_torch)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_torch.parameters(), lr=0.001)

for epoch in range(5):
    model_torch.train()
    total_loss = 0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        outputs = model_torch(x_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

model_torch.eval()

with torch.no_grad():
    outputs = model_torch(x_test_tensor)
    preds = torch.argmax(outputs, dim=1)

accuracy = (preds == y_test_tensor).float().mean()
print("Accuracy:", accuracy.item())


PyTorch Model Architecture:
SimpleNN(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)
Epoch 1, Loss: 0.2925
Epoch 2, Loss: 0.1266
Epoch 3, Loss: 0.0844
Epoch 4, Loss: 0.0647
Epoch 5, Loss: 0.0493
Accuracy: 0.9765999913215637
